In [2]:
import torch
import torch.nn.functional as F
import numpy as np

In [162]:
# x = torch.tensor([[1, 2, 3],
#                   [4, 5, 6]]).float()   # shape [2, 3]

# y = torch.tensor([[7, 8, 9],
#                   [10, 11, 12]]).float()   # shape [2, 3]

In [196]:
x = torch.randint(high=10, size=(2,5)).float()
y = torch.randint(high=10, size=(3,5)).float()
print(x)
print()
print(y)

tensor([[0., 1., 1., 3., 7.],
        [6., 7., 9., 4., 9.]])

tensor([[0., 9., 2., 3., 4.],
        [8., 3., 8., 8., 2.],
        [7., 3., 2., 7., 7.]])


In [197]:
x@y.t()

tensor([[ 48.,  49.,  75.],
        [129., 191., 172.]])

In [198]:
prod = (x.unsqueeze(1)*y.unsqueeze(0)).sum(dim=-1)
prod

tensor([[ 48.,  49.,  75.],
        [129., 191., 172.]])

In [200]:
# ovdsat?
# x_norm_ovdsat = x / torch.norm(x.unsqueeze(1), dim=-1, keepdim=True) # normalize dimension D
# y_norm_ovdsat = y / torch.norm(y.unsqueeze(0), dim=-1, keepdim=True) # normalize dimension D

x_norm_ovdsat = x / x.norm(dim=-1, keepdim=True)
y_norm_ovdsat = y / y.norm(dim=-1, keepdim=True)

print(x_norm_ovdsat)
print(y_norm_ovdsat)

tensor([[0.0000, 0.1291, 0.1291, 0.3873, 0.9037],
        [0.3700, 0.4316, 0.5550, 0.2467, 0.5550]])
tensor([[0.0000, 0.8581, 0.1907, 0.2860, 0.3814],
        [0.5587, 0.2095, 0.5587, 0.5587, 0.1397],
        [0.5534, 0.2372, 0.1581, 0.5534, 0.5534]])


In [202]:
# CoOp
# print(x.norm(dim=-1, keepdim=True))
# print(y.norm(dim=-1, keepdim=True))

x_norm_coop = x / x.norm(dim=-1, keepdim=True)
y_norm_coop = y / y.norm(dim=-1, keepdim=True)

print(x_norm_coop)
print(y_norm_coop)

tensor([[0.0000, 0.1291, 0.1291, 0.3873, 0.9037],
        [0.3700, 0.4316, 0.5550, 0.2467, 0.5550]])
tensor([[0.0000, 0.8581, 0.1907, 0.2860, 0.3814],
        [0.5587, 0.2095, 0.5587, 0.5587, 0.1397],
        [0.5534, 0.2372, 0.1581, 0.5534, 0.5534]])


In [204]:
# ovdsat
sim_ovdsat = (x_norm_ovdsat.unsqueeze(1) * y_norm_ovdsat.unsqueeze(0)).sum(dim=-1)
print(sim_ovdsat)

tensor([[0.5908, 0.4418, 0.7655],
        [0.7584, 0.8226, 0.8385]])


In [206]:
# coop
sim_coop = x_norm_coop @ y_norm_coop.t()
print(sim_coop)

tensor([[0.5908, 0.4418, 0.7655],
        [0.7584, 0.8226, 0.8385]])


### Compare cosine similarity computation

#### OVDSAT

In [13]:
B = 20
K = 1849
D = 1024
N = 220
target_size = 602
patch_2d_size = int(np.sqrt(K))
print('patch_2d_size:', patch_2d_size)

patch_2d_size: 43


In [14]:
feats = torch.rand(B, K, D)
prot = torch.rand(N, D)
print(feats.shape)
print(prot.shape)

torch.Size([20, 1849, 1024])
torch.Size([220, 1024])


In [11]:
feats_reshaped = feats.view(B, 1, K, D) # using view messes it up
prot_reshaped = prot.view(1, N, 1, D)
print(feats_reshaped.shape)
print(prot_reshaped.shape)

RuntimeError: shape '[1, 220, 1, 1024]' is invalid for input of size 168960

In [151]:
# # test dimensions
# print((feats_reshaped * prot_reshaped).shape) # broadcasts to get element-wise product of each row in x with each col of y (D products for each row-col pair)
# print((feats_reshaped * prot_reshaped).sum(dim=3).shape) # sums over D elements for each row-col pair to get dot product

In [7]:
normalize = True

dot_product = (feats_reshaped * prot_reshaped).sum(dim=3)

# In ovdsat, normalize happens after dot product 
if normalize:
    feats_norm = torch.norm(feats_reshaped, dim=3, keepdim=True).squeeze(-1) # normalize dimension D
    prot_norm = torch.norm(prot_reshaped, dim=3, keepdim=True).squeeze(-1) # normalize dimension D
    dot_product /= (feats_norm * prot_norm + 1e-8)
    
print(dot_product.shape)

torch.Size([1, 220, 1849])


To match coop, take the norm first and don't use .view to reshape!!!

In [18]:
# feats_norm = feats_reshaped / feats_reshaped.norm(dim=-1, keepdim=True)
# prot_norm = prot_reshaped / prot_reshaped.norm(dim=-1, keepdim=True)

# print(feats_norm.shape)
# print(prot_norm.shape)

#dot_product = (feats_norm * prot_norm).sum(dim=-1)

feats_norm = (feats / feats.norm(dim=-1, keepdim=True)).squeeze()
prot_norm = prot / prot.norm(dim=-1, keepdim=True)

#dot_product = (feats_norm.unsqueeze(1) * prot_norm.unsqueeze(0)).sum(dim=-1) # This also works!
dot_product = feats_norm @ prot_norm.t()
    
print(dot_product.shape)

torch.Size([20, 1849, 220])


In [16]:
ovdsat_sim = dot_product.squeeze()
print(ovdsat_sim.shape)

torch.Size([20, 1849, 220])


In [288]:
# cosim_list = []

# # Append the similarity scores for this batch to the list
# cosim_list.append(dot_product)

# # Concatenate the similarity scores from different batches
# cosim = torch.cat(cosim_list, dim=1)

# # Reshape to 2D and return class similarity maps
# cosim = cosim.reshape(-1, N, patch_2d_size, patch_2d_size)

# # Interpolate cosine similarity maps to original resolution
# cosim = F.interpolate(cosim, size=target_size, mode='bicubic')

#### CoOp

In [289]:
# reshape feats and prot to match output of image/text encoders for clip
feats_clip = feats.view(K, D)
prot_clip = prot
print(feats_clip.shape)
print(prot_clip.shape)

torch.Size([1849, 1024])
torch.Size([220, 1024])


In [290]:
# normalized features
image_features = feats_clip / feats_clip.norm(dim=-1, keepdim=True)
text_features = prot_clip / prot_clip.norm(dim=-1, keepdim=True)

# cosine similarity as logits
logits_per_image = image_features @ text_features.t() # in CoOp just use this one
logits_per_text = text_features @ image_features.t() # logits_per_text == logits_per_image.T

In [291]:
coop_sim = logits_per_image
print(coop_sim.shape)

torch.Size([1849, 220])


In [292]:
print(logits_per_text.shape)

torch.Size([220, 1849])


In [293]:
print((logits_per_image == logits_per_text.t()).all())
print((logits_per_text == logits_per_image.t()).all())

tensor(True)
tensor(True)


#### Compare result

In [294]:
print(coop_sim.shape)
print(ovdsat_sim.shape)

#print((coop_sim == ovdsat_sim).all())
all_close = torch.allclose(coop_sim, ovdsat_sim, atol=1e-6, rtol=1e-5)
print(all_close)

torch.Size([1849, 220])
torch.Size([1849, 220])
True


In [284]:
print(coop_sim[:5, :5])
print()
print(ovdsat_sim[:5, :5])

tensor([[0.7431, 0.7420, 0.7473, 0.7584, 0.7634],
        [0.7574, 0.7416, 0.7583, 0.7486, 0.7675],
        [0.7449, 0.7348, 0.7603, 0.7507, 0.7767],
        [0.7505, 0.7421, 0.7496, 0.7314, 0.7603],
        [0.7464, 0.7614, 0.7754, 0.7596, 0.7631]])

tensor([[0.7431, 0.7420, 0.7473, 0.7584, 0.7634],
        [0.7574, 0.7416, 0.7583, 0.7486, 0.7675],
        [0.7449, 0.7348, 0.7603, 0.7507, 0.7767],
        [0.7505, 0.7421, 0.7496, 0.7314, 0.7603],
        [0.7464, 0.7614, 0.7754, 0.7596, 0.7631]])


#### Compare norms

In [97]:
norm_coop = feats_clip.norm(dim=-1, keepdim=True).squeeze()
norm_coop.shape

torch.Size([1849])

In [98]:
norm_ovdsat = torch.norm(feats_reshaped, dim=3, keepdim=True).squeeze()
#norm_ovdsat = feats_reshaped.norm(dim=-1, keepdim=True).squeeze()
norm_ovdsat.shape

torch.Size([1849])

In [99]:
print((norm_coop == norm_ovdsat).all())

tensor(True)
